In [1]:
# import needed libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.signal import argrelextrema

from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [28]:
# define ODE model
def goodwin_meanfield(X, t, parameters):

    N = parameters['N']

    # parameters
    p1 = parameters['p1']
    p2 = parameters['p2']
    p3 = parameters['p3']
    p4 = parameters['p4']
    d1 = parameters['d1']
    d2 = parameters['d2']      # array (heterogeneous)
    d3 = parameters['d3']
    d4 = parameters['d4']
    K = parameters['K']
    Kc = parameters['Kc']
    h = parameters['h']

    # reshape state vector
    X = X.reshape((N,4))
    
    x = X[:,0]
    y = X[:,1]
    z = X[:,2]
    v = X[:,3]

    # mean field
    MF = np.mean(v)

    dXdt = np.zeros_like(X)

    for i in range(N):

        dxdt = p1*K**h/(K**h + z[i]**h) - d1*x[i]/(K+x[i]) + Kc*MF/(K + Kc*MF)
        dydt = p2*x[i] - d2[i]*y[i]/(K+y[i])
        dzdt = p3*y[i] - d3*z[i]/(K+z[i])
        dvdt = p4*x[i] - d4*v[i]/(K+v[i])

        dXdt[i] = [dxdt, dydt, dzdt, dvdt]

    return dXdt.flatten()

In [ ]:
def goodwin_plot(N=10, Kc=0.0):

    # time
    dt = 0.01
    t = np.arange(0,2500,dt)
    n_days_plot = 10

    # heterogeneous d2
    np.random.seed(20260312)
    d2 = 0.35 + 0.02*np.random.randn(N)

    # parameters
    parameters = {
        'N':N,
        'p1':0.7,
        'p2':0.7,
        'p3':0.7,
        'p4':0.35,
        'd1':0.35,
        'd2':d2,
        'd3':0.35,
        'd4':1,
        'K':1,
        'Kc':Kc,
        'h':4
    }

    # initial conditions
    X0 = np.random.rand(N,4)*0.1
    X0 = X0.flatten()

    # solve ODE
    sol = odeint(goodwin_meanfield, X0, t, args=(parameters,))
    sol = sol.reshape(len(t), N, 4)

    x = sol[:,:,0]
    y = sol[:,:,1]
    z = sol[:,:,2]
    v = sol[:,:,3]

    MF = np.mean(v, axis=1)

    # plotting
    fig = plt.figure(figsize=(11,10))

    # time series
    ax1 = fig.add_subplot(221)
    ax2 = fig.add_subplot(222)
    ax3 = fig.add_subplot(223)
    ax4 = fig.add_subplot(224)

    for i in range(N):
        ax3.plot(t/24, v[:,i], alpha=0.7)
        ax1.plot(t[-int(24*n_days_plot/dt):]/24, v[-int(24*n_days_plot/dt):,i], alpha=0.5, lw=0.75)

    ax4.plot(t/24, MF, 'k', linewidth=3, label='mean field')
    ax2.plot(t[-int(24*n_days_plot/dt):]/24, MF[-int(24*n_days_plot/dt):], 'k', 
             linewidth=3, label='mean field')

    ax1.set_xlabel('time (days)'); ax3.set_xlabel('time (days)')
    ax1.set_ylabel('v'); ax3.set_ylabel('v')
    ax1.set_title('Individual oscillators')
    ax1.grid(); ax1.grid()

    ax2.set_xlabel('time (days)'); ax4.set_xlabel('time (days)')
    ax2.set_title('Mean Field')
    ax4.set_ylabel('mean field'); ax2.set_ylabel('mean field')
    ax4.grid(); ax2.grid()
    ax1.set_ylim([0, 0.2]); ax2.set_ylim([0, 0.2])

    plt.suptitle(f"N={N}, coupling={Kc}")

widget = interactive(
    goodwin_plot,
    N=(10,50,5),
    Kc=(0,0.5,0.025)
)

display(widget)

interactive(children=(IntSlider(value=10, description='N', max=50, min=10, step=5), FloatSlider(value=0.0, des…